In [ ]:
import re
import sys
import numpy as np
from pathlib import Path as ph
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
# Add parent directory to sys.path
sys.path.append(str(ph().resolve().parent))
from src.functions.runtime import from_file, towa_file, get_directory_files

In [ ]:
runtime_path_inp = input("Enter the runtime path ('same','<path>'): ").strip().lower()
runtime_uuid_inp = input("Enter the model configuration ('<uuid>','all'): ").strip().lower()

In [ ]:
########################
# Runtime variables
########################

if runtime_path_inp == "same":
    runtime_path = "."
else:
    runtime_path = runtime_path_inp

runtime_uuid = runtime_uuid_inp

configs_path = f"{runtime_path}/configs"
tokenizers_path = f"{runtime_path}/tokenizers"
inputs_path = f"{runtime_path}/inputs"
outputs_path = f"{runtime_path}/outputs"
models_path = f"{runtime_path}/models"
charts_path = f"{runtime_path}/charts"
statistics_path = f"{runtime_path}/statistics"

In [ ]:
########################
# Add row experiment file components
########################

def process_add_row_expirement_file_components(folder_path, file_prefix, runtime_uuid):
    runtime_uuids = []
    if runtime_uuid == "all":
        # If a all is provided, add ithem all to the list
        runtime_uuids = get_directory_files(folder_path, file_prefix)
    else:
        # If a specific UUID is provided, add it to the list
        runtime_uuids.append(runtime_uuid)

    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    # Loop through each GUID to extract weights
    for runtime_uuid in runtime_uuids:
        print(runtime_uuid)
        # Load the model from a single file
        config_path_inp = f"{configs_path}/config_{runtime_uuid}.json"
        config_json = from_file(config_path_inp, "json")
        report_path_inp = f"{outputs_path}/report_{runtime_uuid}.json"
        report_json = from_file(report_path_inp, "json")
        completion_path_inp = f"{outputs_path}/completion_{runtime_uuid}.json"
        completion_json = from_file(completion_path_inp, "json")

        row = [
            runtime_uuid,
            config_json["runtime"]["model_version"],
            config_json["runtime"]["model_params"],
            config_json["runtime"]["model_size"],
            config_json["runtime"]["device_name"],
            config_json["runtime"]["library_driver"],
            config_json["c_device"],
            config_json["c_tokenizer"],
            config_json["c_sequence"],
            config_json["c_attention"],
            config_json["c_network"],
            config_json["n_ctx"],
            config_json["n_emb"],
            config_json["r_dropout"],
            config_json["s_head"],
            config_json["n_heads"],
            config_json["n_layers"],
            config_json["n_epochs"],
            config_json["s_batch"],
            config_json["r_learn"],
            report_json["dataset"],
            f"{report_json["avg_train_time_per_batch"]:.2f}",
            f"{report_json["avg_val_time_per_batch"]:.2f}",
            f"{report_json["average_time_per_epoch"]:.0f}",
            f"{report_json["total_time"]:.0f}",
            f"{completion_json["inference_total_time"]:.0f}",
            f"{completion_json["evaluation"]:.1f}/100"
        ]
        print(f"{'-'*10} {'Adding Row'} {'-'*10}")
        print(row)
        statistic.append(row)

        # Reconstruct the statistic with header and added data
        towa_file(statistic_path, "csv", statistic)

In [ ]:
########################
# Add column experiment file components
########################

def process_add_column_expirement_file_components(folder_path, file_prefix, runtime_uuid):
    runtime_uuids = []
    if runtime_uuid == "all":
        # If a all is provided, add ithem all to the list
        runtime_uuids = get_directory_files(folder_path, file_prefix)
    else:
        # If a specific UUID is provided, add it to the list
        runtime_uuids.append(runtime_uuid)

    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    # Target index (e.g., insert at position 3)
    target_index = 22

    # Insert into header
    new_column_name = "val_batch_time_execution"
    statistic[0].insert((target_index-1), new_column_name)

    # Loop through each GUID to extract weights
    for runtime_uuid in runtime_uuids:
        print(runtime_uuid)
        # Load the model from a single file
        config_path_inp = f"{configs_path}/config_{runtime_uuid}.json"
        config_json = from_file(config_path_inp, "json")
        report_path_inp = f"{outputs_path}/report_{runtime_uuid}.json"
        report_json = from_file(report_path_inp, "json")
        completion_path_inp = f"{outputs_path}/completion_{runtime_uuid}.json"
        completion_json = from_file(completion_path_inp, "json")

        print(f"{'-'*10} {'Adding Column'} {'-'*10}")

        # Insert into each data row
        for i in range(1, len(statistic)):
            if statistic[i][0] == runtime_uuid:
                statistic[i].insert((target_index-1), f"{float(report_json["avg_val_time_per_batch"]):.2f}")

        # Reconstruct the statistic with header and added data
        towa_file(statistic_path, "csv", statistic)

In [ ]:
########################
# Edit column experiment file components
########################

def process_edit_static_column_expirement_file_components(folder_path, file_prefix, runtime_uuid):
    # Load the model from a single file
    config_path_inp = f"{configs_path}/config_{runtime_uuid}.json"
    config_json = from_file(config_path_inp, "json")
    report_path_inp = f"{outputs_path}/report_{runtime_uuid}.json"
    report_json = from_file(report_path_inp, "json")
    completion_path_inp = f"{outputs_path}/completion_{runtime_uuid}.json"
    completion_json = from_file(completion_path_inp, "json")
    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    # Extract header and data
    header = statistic[0]
    data = statistic[1:]

    # Define target UUID and column to update
    column_name_to_update = "model_size"
    column_value_to_update = "XL"

    # Find column index
    col_index = header.index(column_name_to_update)

    print(f"{'-'*10} {'Updating Row'} {'-'*10}")
    print(f"Updating column '{column_name_to_update}' for UUID '{runtime_uuid}'")

    # Update the value for the matching UUID
    for row in data:
        if row[header.index("baby's brain")] == runtime_uuid or runtime_uuid == "all":
            row[col_index] = column_value_to_update
            break # Remove this if multiple rows share the same UUID

    # Reconstruct the statistic with header and updated data
    towa_file(statistic_path, "csv", statistic)

In [ ]:
########################
# Edit column experiment file components
########################

def process_edit_dynamic_column_expirement_file_components(folder_path, file_prefix, runtime_uuid):
    # Load the model from a single file
    config_path_inp = f"{configs_path}/config_{runtime_uuid}.json"
    config_json = from_file(config_path_inp, "json")
    report_path_inp = f"{outputs_path}/report_{runtime_uuid}.json"
    report_json = from_file(report_path_inp, "json")
    completion_path_inp = f"{outputs_path}/completion_{runtime_uuid}.json"
    completion_json = from_file(completion_path_inp, "json")
    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    # Extract header and data
    header = statistic[0]
    header[0] = header[0].lstrip('\ufeff').lstrip('\ufeff') # Remove BOM if present
    data = statistic[1:]

    # Find columns index
    col_index_a = header.index("model_params")
    col_index_b = header.index("model_size")

    print(f"{'-'*10} {'Updating Rows'} {'-'*10}")

    # Update the value for the matching UUID
    for row in data:
        if row[header.index("baby's brain")] == runtime_uuid or runtime_uuid == "all":
            row[col_index_b] = (int(row[col_index_a]) * 4)
            break # Remove this if multiple rows share the same UUID

    # Reconstruct the statistic with header and updated data
    towa_file(statistic_path, "csv", statistic)

In [ ]:
def process_visualize_file_components(folder_path, file_prefix, runtime_uuid):
    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    # Extract header and data
    header = statistic[0]
    header[0] = header[0].lstrip('\ufeff').lstrip('\ufeff') # Remove BOM if present
    data = statistic[1:]

    # Find columns index    
    w_col_a = header.index("c_device")
    w_col_b = header.index("n_layers")
    x_col_a = header.index("baby's brain")
    x_col_b = header.index("c_attention")
    x_col_c = header.index("c_network")
    x_col_d = header.index("total_time_execution")
    y_col = header.index("inference_quality_execution")

    n_layers = [4,8,16]
    fig, axes = plt.subplots(len(n_layers), 1, figsize=(16, 8 * len(n_layers)))  # Removed sharex=True
    if len(n_layers) == 1:
        axes = [axes]  # Ensure axes is always a list

    for idx, n_layer in enumerate(n_layers):
        x_axes = []
        y_axes = []
        colors = []
        x_part2_list = []
        for row in data:
            # Filter data
            if str(row[w_col_a]) == "gpu" and int(row[w_col_b]) == n_layer:
                x_part1 = str(row[x_col_a])
                x_part2 = f"{str(row[x_col_b])}\n{str(row[x_col_c])}\n{(int(row[x_col_d])/3600):.0f}h"
                x_axes.append(x_part1)
                y_axes.append(float(row[y_col].replace("/100", "")))
                x_part2_list.append(x_part2)
                if "finetuned" in x_part1:
                    colors.append("red")
                else:
                    colors.append("green")
        
        ax = axes[idx]
        bars = ax.bar(x_axes, y_axes, color=colors, edgecolor='black')
        legend_handles = [
            Patch(color='red', label='finetuned'),
            Patch(color='green', label='original')
        ]
        ax.legend(handles=legend_handles)
        for bar, label in zip(bars, x_part2_list):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() / 2,
                label,
                ha='center',
                va='center',
                rotation=0,
                color='white',
                fontsize=8,
                fontweight='bold'
            )
        ax.set_title(f"Model Quality for n_layers={n_layer}")
        ax.set_ylabel("Quality")
        ax.grid(True)
        ax.set_xticks(range(len(x_axes)))
        ax.set_xticklabels(x_axes, rotation=90, ha='center')  # This now works for every row

    for ax in axes:
        ax.set_xlabel("Model")

    plt.tight_layout()
    plt.savefig(f"{statistics_path}/statistic_expirements_layers.png", bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# === Run the script ===
folder_path = f"{runtime_path}/outputs"
file_prefix = f"report"
_ = get_directory_files(folder_path, file_prefix)
process_add_row_expirement_file_components(folder_path, file_prefix, runtime_uuid)